# Finz Analysis Model — Training on Colab

**GPU**: Runtime → Change runtime type → T4 GPU

이 노트북은 다음을 수행합니다:
1. 레포 클론 & 의존성 설치
2. 뉴스 + 주가 데이터 수집 (46개 종목, 15년치)
3. 모델 학습 (DistilBERT + Cross-Attention Fusion)
4. 학습된 모델 다운로드

## 0. GPU 확인

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU가 없습니다! Runtime → Change runtime type → T4 GPU 를 선택하세요.")

## 1. 레포 클론 & 설치

In [ ]:
!git clone https://github.com/IkhyunKwon-00/finzanalysismodel.git
%cd finzanalysismodel
!pip install -e . -q

## 2. 데이터 수집

Yahoo Finance에서 46개 종목 뉴스 + 15년치(2011~) 주가/매크로 데이터를 수집합니다.

In [ ]:
!python -m scripts.collect_data

### (선택) Financial PhraseBank 추가

학습 데이터를 ~5,000개 전문가 라벨링 문장으로 보강합니다.

In [ ]:
# Financial PhraseBank 다운로드 (선택사항 — 학습 데이터 보강)
# Hugging Face에서 다운받아 data/raw/에 배치
try:
    from datasets import load_dataset
    ds = load_dataset("financial_phrasebank", "sentences_allagree", trust_remote_code=True)
    with open("data/raw/financial_phrasebank.txt", "w") as f:
        for row in ds["train"]:
            label_map = {0: "negative", 1: "neutral", 2: "positive"}
            f.write(f"{row['sentence']}@{label_map[row['label']]}\n")
    print(f"Financial PhraseBank saved: {len(ds['train'])} sentences")
    # 데이터 재수집 (PhraseBank 포함)
    !python -m scripts.collect_data
except Exception as e:
    print(f"PhraseBank 다운로드 실패 (선택사항이므로 무시 가능): {e}")

## 3. 데이터셋 확인

In [ ]:
import pandas as pd

df = pd.read_parquet("data/processed/dataset.parquet")
print(f"총 샘플: {len(df)}")
print(f"컬럼: {list(df.columns)}")
print(f"\n종목 분포:")
print(df["symbol"].value_counts().head(20))
print(f"\n레이블 분포 (30d):")
print(df["label_30d"].value_counts())
df.head(3)

## 4. 모델 학습

T4 GPU 기준 설정:
- batch_size: 32
- max_length: 512
- freeze_layers: 4 (DistilBERT 6개 레이어 중 4개 동결)
- epochs: 30 (early stopping patience=5)

In [ ]:
!python -m src.train --data data/processed/dataset.parquet

## 5. 학습 결과 확인

In [ ]:
import torch
from pathlib import Path

best_path = Path("models/best/model.pt")
if best_path.exists():
    ckpt = torch.load(best_path, map_location="cpu", weights_only=False)
    print(f"Best model — Epoch: {ckpt['epoch']}")
    print(f"Val Loss: {ckpt['val_loss']:.4f}")
    if 'val_metrics' in ckpt:
        for k, v in ckpt['val_metrics'].items():
            print(f"  {k}: {v:.4f}")
else:
    print("모델 파일이 없습니다. 학습이 완료되었는지 확인하세요.")

## 6. 추론 테스트

In [ ]:
import torch
from transformers import AutoTokenizer
from src.config import load_config
from src.models.momentum_model import FinzMomentumModel

cfg = load_config()
ckpt = torch.load("models/best/model.pt", map_location="cpu", weights_only=False)

model = FinzMomentumModel(
    num_numerical_features=ckpt["num_numerical_features"],
    text_model_name=cfg["text_encoder"]["model_name"],
    text_embed_dim=cfg["model"]["text_embed_dim"],
    numerical_hidden=cfg["model"]["numerical_hidden"],
    fusion_hidden=cfg["model"]["fusion_hidden"],
    num_heads=cfg["model"]["num_heads"],
    dropout=0.0,
    num_horizons=cfg["model"]["num_horizons"],
)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

tokenizer = AutoTokenizer.from_pretrained(cfg["text_encoder"]["model_name"])

# 테스트 뉴스
text = "Tesla reports record quarterly revenue on strong EV demand [SEP] Electric vehicle maker Tesla Inc reported record revenue of $25 billion in Q4."
enc = tokenizer(text, max_length=cfg["text_encoder"]["max_length"], padding="max_length", truncation=True, return_tensors="pt")
numerical = torch.zeros(1, ckpt["num_numerical_features"])

with torch.no_grad():
    outputs = model(enc["input_ids"], enc["attention_mask"], numerical)

LABELS = {0: "negative", 1: "neutral", 2: "positive"}
horizons = [30, 180, 360]
for i, h in enumerate(horizons):
    logits = outputs[f"logits_{i}"][0]
    probs = torch.softmax(logits, dim=-1)
    pred = probs.argmax().item()
    ret = outputs[f"return_{i}"][0].item()
    print(f"  {h}d: {LABELS[pred]} (conf={probs[pred].item()*100:.1f}%) | return={ret:.2f}%")

## 7. Sentence Attribution (핵심 문장 선별)

In [ ]:
from src.models.sentence_attribution import get_sentence_attributions

text = "Tesla reports record quarterly revenue on strong EV demand. Electric vehicle maker Tesla Inc reported record revenue of $25 billion in Q4. The company also announced plans to expand its Gigafactory in Texas."
enc = tokenizer(text, max_length=cfg["text_encoder"]["max_length"], padding="max_length", truncation=True, return_tensors="pt")
numerical = torch.zeros(1, ckpt["num_numerical_features"])

results = get_sentence_attributions(
    model=model,
    tokenizer=tokenizer,
    text=text,
    input_ids=enc["input_ids"],
    attention_mask=enc["attention_mask"],
    numerical=numerical,
    target_horizon=0,
    n_steps=30,
    top_k=5,
)

print("핵심 문장 (Integrated Gradients 기반):")
for r in results:
    print(f"  #{r['rank']} (score={r['attribution_score']:.1f}) {r['sentence']}")

## 8. 모델 다운로드

학습된 모델을 로컬로 다운로드합니다.

In [ ]:
from google.colab import files

# 최적 모델 다운로드
files.download("models/best/model.pt")
print("✅ model.pt 다운로드 완료 — 이 파일을 finzanalysismodel/models/best/ 에 배치하세요.")

### (선택) Google Drive에 저장

In [ ]:
# Google Drive 마운트 후 저장
from google.colab import drive
drive.mount("/content/drive")

import shutil
save_dir = "/content/drive/MyDrive/finz_model/"
!mkdir -p "{save_dir}"
shutil.copy("models/best/model.pt", save_dir)
print(f"✅ 모델이 Google Drive에 저장됨: {save_dir}model.pt")